In [10]:
import torch
from transformers import AutoModelForCausalLM
from janus.models import MultiModalityCausalLM, VLChatProcessor
from janus.utils.io import load_pil_images
import os
import math
import fitz  # For PDF text extraction
from PIL import Image

# Initialize Janus model
model_path = "deepseek-ai/Janus-Pro-1B"
vl_chat_processor = VLChatProcessor.from_pretrained(model_path)
tokenizer = vl_chat_processor.tokenizer

# Load model for CPU
vl_gpt = AutoModelForCausalLM.from_pretrained(
    model_path,
    trust_remote_code=True
).to(torch.bfloat16).cpu().eval()

def extract_text_from_pdf(pdf_path):
    """Extract text content from a PDF file."""
    document = fitz.open(pdf_path)
    text = ""
    for page_num in range(len(document)):
        page = document.load_page(page_num)
        text += page.get_text()
    return text

def create_collage(image_paths, thumb_width=384, thumb_height=384, grid_columns=3):
    """
    Create a collage from multiple images.
    
    Parameters:
        image_paths (list): List of paths to image files.
        thumb_width (int): Width to resize each image.
        thumb_height (int): Height to resize each image.
        grid_columns (int): Number of columns in the collage grid.
        
    Returns:
        A single PIL Image object that is the collage of all images.
    """
    images = [Image.open(path).resize((thumb_width, thumb_height)) for path in image_paths]
    grid_rows = math.ceil(len(images) / grid_columns)
    collage_width = grid_columns * thumb_width
    collage_height = grid_rows * thumb_height
    collage = Image.new('RGB', (collage_width, collage_height), color=(255, 255, 255))
    for index, img in enumerate(images):
        row = index // grid_columns
        col = index % grid_columns
        collage.paste(img, (col * thumb_width, row * thumb_height))
    return collage

def prepare_conversation(prompt, images):
    """
    Prepare the conversation structure for the Janus model.
    
    Parameters:
        prompt (str): The text prompt.
        images (list): A list of PIL Image objects.
        
    Returns:
        A list representing the conversation.
    """
    return [
        {
            "role": "<|User|>",
            "content": f"<image_placeholder>\n{prompt}",
            "images": images,
        },
        {"role": "<|Assistant|>", "content": ""},
    ]

def generate_response(conversation):
    """Generate a response using the Janus model."""
    # Load and process images from the conversation
    pil_images = [img for img in conversation[0]["images"]]
    
    # Prepare inputs for the model
    prepare_inputs = vl_chat_processor(
        conversations=conversation,
        images=pil_images,
        force_batchify=True
    ).to("cpu")

    # Generate embeddings and run the model
    inputs_embeds = vl_gpt.prepare_inputs_embeds(**prepare_inputs)
    outputs = vl_gpt.language_model.generate(
        inputs_embeds=inputs_embeds,
        attention_mask=prepare_inputs.attention_mask,
        pad_token_id=tokenizer.eos_token_id,
        bos_token_id=tokenizer.bos_token_id,
        eos_token_id=tokenizer.eos_token_id,
        max_new_tokens=1000,
        do_sample=False,
        use_cache=True,
    )

    return tokenizer.decode(outputs[0].cpu().tolist(), skip_special_tokens=True)

def get_response_with_images(question, image_paths):
    """
    Process a request with multiple images by creating a collage.
    
    Parameters:
        question (str): The input text prompt.
        image_paths (list): List of image file paths.
        
    Returns:
        The model's generated response.
    """
    collage = create_collage(image_paths)
    conversation = prepare_conversation(question, [collage])
    return generate_response(conversation)

def get_response_with_pdf_and_images(question, pdf_path, image_paths):
    """
    Process a request that includes a PDF and images.
    
    Parameters:
        question (str): The input text prompt.
        pdf_path (str): Path to the PDF file.
        image_paths (list): List of image file paths.
        
    Returns:
        The model's generated response.
    """
    pdf_text = extract_text_from_pdf(pdf_path)
    full_prompt = f"{question}\n\nPDF Content:\n{pdf_text}"
    collage = create_collage(image_paths)
    conversation = prepare_conversation(full_prompt, [collage])
    return generate_response(conversation)

# File paths configuration
input_text_file_path = 'Sample to rate/Text/input.txt'
report_file_path = 'Sample to rate/Text/output.txt'
input_example_file_path = 'Examples/Text/input_text.txt'
output_example_files = [
    'Examples/Text/Example1.txt',
    'Examples/Text/Example2.txt',
    'Examples/Text/Example3.txt',
    'Examples/Text/Example4.txt'
]

# Read input files
with open(input_text_file_path, 'r', encoding='utf-8') as f:
    input_text = f.read()

with open(report_file_path, 'r', encoding='utf-8') as f:
    report = f.read()

with open(input_example_file_path, 'r', encoding='utf-8') as f:
    input_example = f.read()

# Read example outputs
example_outputs = []
for path in output_example_files:
    with open(path, 'r', encoding='utf-8') as f:
        example_outputs.append(f.read())

# Construct evaluation question
explanations = "followed by your step-by-step reasoning"
question = f"""Your task is to rate a report based on its coverage with respect to an input context and input plots. Coverage is on a scale from 1 (worst) to 5 (perfect). Your answer must state the number you give for coverage {explanations}.
Consider the following examples. For example, the input context is: {input_example} and the input plot is given to you as an attachment and tagged with the label "Image used for the example".
An example of a report is: {example_outputs[0]}. This example scores 5 on coverage, because it only provided information that was in the input (no hallucinations). Another example of a report is: {example_outputs[1]}. This example scores 3 on coverage: it lost two points because it gave two pieces of information that were not in the input (car brand, model commissioner). A third report is: {example_outputs[2]}. This example scores 5 on coverage, because it only provided information that was in the input (no hallucinations). A last example is: {example_outputs[3]}. This last example scores 3 on coverage: it lost two points because it gave two pieces of information that were not in the input (car brand, model commissioner).
In your case, the input context is: {input_text} The accompanying input plots are given to you as an attachment. 
The report that you must rate for coverage is {report}"""

# Prepare image paths
image_directory_1 = 'Sample to rate/Images/Milk'
image_directory_2 = 'Examples/Images'
image_paths = [os.path.join(image_directory_1, f'Picture {i}.png') for i in range(1, 11)]
image_paths.append(os.path.join(image_directory_2, 'Picture Example.png'))

# Generate and print response
response = get_response_with_images(question, image_paths)
print("Model Evaluation:")
print(response)

Some kwargs in processor config are unused and will not have any effect: image_tag, num_image_tokens, mask_prompt, ignore_id, add_special_token, sft_format. 


Model Evaluation:
The agent-based model aims to reproduce adoption behaviors of milk consumption by the British public from 1974 to 2005. The goal is to explore the influence of perception, habits, and social influence in an individual's decision-making process regarding milk choice. The study conducts an experiment to compare the performance of two model variants in reproducing observed milk consumption trends. Agents represent adult consumers who occupy a random position in an information environment. Each agent has a disposition to consider alternative milk choices, based on the tested mechanisms of how agents become disposed to consider their choices. Agents make milk selection choices based on a function for each alternative, composed of perceived health and environmental characteristics, modified by habit, social influence, and evaluation of previous choices. If the disposition requirement has been met, consumption of each milk type is split proportionately by the choice function